# Neural rescoring with InstaNovo as a learned score function

This notebook re-purposes a pretrained **InstaNovo** transformer (originally trained for de novo peptide sequencing) as a learned **scoring function** for database-search-style PSM rescoring — analogous to **Casanovo-DB** (Ananth & Sanders et al., *Bioinformatics* 2024).

## Idea

Given a spectrum and a candidate peptide $s_1, s_2, \ldots, s_n$, we run the InstaNovo decoder under teacher forcing and read out the model's per-residue log-probabilities $\log p(s_i \mid s_{<i}, \text{spectrum})$. The Casanovo-DB score is the **mean log-probability** (i.e. the log of the geometric mean of the per-residue probabilities):

$$
\text{score}(\text{spectrum}, s) \;=\; \frac{1}{n} \sum_{i=1}^{n} \log p(s_i \mid s_{<i}, \text{spectrum})
$$

The geometric mean (versus the arithmetic mean of probabilities) penalises low-confidence positions more harshly, which the paper found to substantially improve calibration in the database-search setting.

## What this notebook supports

Inputs are kept flexible — plug in whichever of these you have:

1. **MGF** spectra + candidate peptides (CSV: `spectrum_id, peptide, [decoy]`).
2. **CSV** spectra (mz_array, intensity_array, precursor_mz, precursor_charge) + candidates.
3. **FASTA** protein database — digested in-notebook (trypsin, configurable) into candidates.
4. **Pre-computed encoder embeddings** (the top-k retrieved spectrum embeddings from your retrieval stage) — score directly against candidates without re-running the encoder.

All four paths converge on the same `casanovo_db_score(...)` core.

> **InstaNovo predicts peptides right-to-left.** The scoring helper takes care of reversing the candidate sequence before encoding, matching the model's training distribution. Don't reverse it yourself.

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the local InstaNovo package importable when the notebook is run from the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INSTANOVO_PATH = PROJECT_ROOT / "InstaNovo"
if str(INSTANOVO_PATH) not in sys.path:
    sys.path.insert(0, str(INSTANOVO_PATH))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from instanovo.transformer.model import InstaNovo
from instanovo.transformer.data import TransformerDataProcessor
from instanovo.utils.data_handler import SpectrumDataFrame
from instanovo.utils.residues import ResidueSet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 2. Load the pretrained InstaNovo model

Downloads + caches `instanovo-v1.2.0` on first run via `from_pretrained`.

In [ ]:
MODEL_ID = "instanovo-v1.2.0"

model, model_config = InstaNovo.from_pretrained(MODEL_ID)
model = model.to(DEVICE).eval()

residue_set: ResidueSet = model.residue_set
print(f"Loaded {MODEL_ID} | vocab size = {model.vocab_size}")

## 3. Core: Casanovo-DB-style scoring

The function below takes either a **batch of spectra** (raw peaks + precursor) or a **batch of pre-computed encoder embeddings**, plus a **list of candidate peptide strings** per spectrum, and returns one Casanovo-DB log-score per (spectrum, candidate) pair.

Implementation notes:
- Runs the encoder **once per spectrum** (or skips it if `spectra_embedding` is supplied), then replicates the embedding for each candidate — cheap.
- Teacher-forces the (reversed) candidate through the decoder and reads out the gathered per-position log-probabilities.
- Computes the mean log-prob over non-padded positions (= log of the geometric mean of probs), exactly as in Casanovo-DB.

In [ ]:
@torch.no_grad()
def encode_candidates(
    candidates: list[str],
    residue_set: ResidueSet,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Tokenize + reverse + EOS-pad a list of candidate peptide strings.

    Returns:
        peptides: LongTensor (C, L) padded with PAD_INDEX. Reversed (right-to-left) and EOS-terminated.
        peptides_mask: BoolTensor (C, L), True at PAD positions.
    """
    encoded = [
        residue_set.encode(
            residue_set.tokenize(p)[::-1],
            add_eos=True,
            return_tensor="pt",
        )
        for p in candidates
    ]
    lengths = torch.tensor([t.shape[0] for t in encoded], dtype=torch.long)
    peptides = torch.nn.utils.rnn.pad_sequence(
        encoded, batch_first=True, padding_value=residue_set.PAD_INDEX
    )
    L = peptides.shape[1]
    peptides_mask = torch.arange(L, dtype=torch.long)[None, :] >= lengths[:, None]
    return peptides.to(device), peptides_mask.to(device)


@torch.no_grad()
def casanovo_db_score(
    model: InstaNovo,
    candidates: list[str],
    *,
    spectra: torch.Tensor | None = None,
    precursors: torch.Tensor | None = None,
    spectra_mask: torch.Tensor | None = None,
    spectra_embedding: torch.Tensor | None = None,
    reduction: str = "mean",
) -> torch.Tensor:
    """Score a list of candidate peptides against ONE spectrum.

    Provide EITHER (spectra, precursors, [spectra_mask]) OR a precomputed `spectra_embedding`.
    `spectra` is shape (1, P, 2); `precursors` is (1, 3) = [mass, charge, m/z].
    `spectra_embedding` is the encoder output (1, T, d_model) returned by `_encoder` / `_flash_encoder`.

    Args:
        reduction: 'mean' (Casanovo-DB, log of geometric mean of per-residue probs)
                   or 'sum' (the joint log-probability of the sequence).

    Returns:
        FloatTensor (C,) of per-candidate scores (log-probabilities).
    """
    if reduction not in {"mean", "sum"}:
        raise ValueError("reduction must be 'mean' or 'sum'")
    if (spectra is None or precursors is None) and spectra_embedding is None:
        raise ValueError("Pass either (spectra, precursors) or spectra_embedding.")

    device = next(model.parameters()).device
    model.eval()

    # 1. Encode the spectrum once (skip if a precomputed embedding is supplied).
    if spectra_embedding is None:
        if model.use_flash_attention:
            spectra_embedding, spectra_mask = model._flash_encoder(spectra.to(device), precursors.to(device))
        else:
            spectra_embedding, spectra_mask = model._encoder(
                spectra.to(device),
                precursors.to(device),
                spectra_mask.to(device) if spectra_mask is not None else None,
            )

    # 2. Encode candidates (reversed + EOS).
    peptides, peptides_mask = encode_candidates(candidates, model.residue_set, device)
    C = peptides.shape[0]

    # 3. Replicate the spectrum embedding once per candidate.
    spec_emb = spectra_embedding.expand(C, -1, -1).contiguous()
    spec_msk = spectra_mask.expand(C, -1).contiguous() if spectra_mask is not None else None

    # 4. Teacher-forced decode + per-residue log-probs.
    if model.use_flash_attention:
        logits = model._flash_decoder(spec_emb, peptides, spec_msk, peptides_mask, add_bos=True)
    else:
        logits = model._decoder(spec_emb, peptides, spec_msk, peptides_mask, add_bos=True)

    log_probs = F.log_softmax(logits, dim=-1)
    # gather log-prob assigned to each true next token (BOS predicts pos 0, ..., pos n-1 predicts EOS).
    seq_logp = torch.gather(log_probs, -1, peptides.unsqueeze(-1)).squeeze(-1)
    seq_logp = seq_logp.masked_fill(peptides_mask, 0.0)

    summed = seq_logp.sum(dim=-1)
    if reduction == "sum":
        return summed.cpu()
    valid = (~peptides_mask).sum(dim=-1).clamp(min=1).float()
    return (summed / valid).cpu()

### Sanity check

Wire up a single (spectrum, candidate) pair and confirm the scorer runs end-to-end. Replace the placeholder peaks with anything from your data; the score is meaningful only relative to other candidates for the same spectrum.

In [ ]:
# Dummy spectrum: 5 random peaks, charge 2, m/z = 500.
_dummy_spectra = torch.tensor([[[100.5, 0.4], [200.1, 0.6], [350.7, 0.3], [488.2, 0.5], [600.9, 0.2]]])
_dummy_precursors = torch.tensor([[(500.0 - 1.00727647) * 2.0, 2.0, 500.0]])  # mass, charge, m/z

scores = casanovo_db_score(
    model,
    candidates=["PEPTIDE", "PEPTIDR", "AAAAAAA"],
    spectra=_dummy_spectra,
    precursors=_dummy_precursors,
    reduction="mean",
)
print("sanity-check scores:", scores.tolist())

## 4. Pipeline: rescore an MGF / CSV file with per-spectrum candidates

Plug in:
- `SPECTRA_PATH` — path to your `.mgf` / `.mzml` / `.mzxml` / `.csv` / `.parquet` file.
- `CANDIDATES_PATH` — a CSV with columns `spectrum_id, peptide, [decoy]` (one row per (spectrum, candidate) pair). The `spectrum_id` must match the spectra source (MGF `TITLE`, CSV `id`, etc.).

If you have the candidates as a Python dict `{spectrum_id: [peptide, ...]}`, skip the CSV load and feed the dict in directly.

In [ ]:
# ----- USER INPUTS -----
SPECTRA_PATH: str | None = None       # e.g. "data/runs/sample.mgf"
CANDIDATES_PATH: str | None = None    # CSV with columns: spectrum_id, peptide, [decoy]
OUT_PATH: str = "rescored_psms.csv"
# -----------------------

processor = TransformerDataProcessor(
    residue_set=residue_set,
    n_peaks=200,
    min_mz=50.0,
    max_mz=2500.0,
    min_intensity=0.01,
    remove_precursor_tol=2.0,
    reverse_peptide=True,
    annotated=False,           # we score externally, no peptide column required
    return_str=False,
    add_eos=True,
)

In [ ]:
def load_spectra_with_ids(path: str) -> tuple[SpectrumDataFrame, list[str]]:
    """Load spectra + return a per-row spectrum_id (TITLE for MGF, falls back to row index)."""
    sdf = SpectrumDataFrame.load(path, lazy=False, is_annotated=False, add_spectrum_id=True)
    df = sdf.to_pandas()
    # Try common id columns; fall back to enumerated index.
    for col in ("spectrum_id", "id", "title", "scans"):
        if col in df.columns:
            ids = df[col].astype(str).tolist()
            return sdf, ids
    return sdf, [str(i) for i in range(len(df))]


def rescore_from_files(
    spectra_path: str,
    candidates_path: str,
    *,
    reduction: str = "mean",
) -> pd.DataFrame:
    """Rescore (spectrum_id, peptide) pairs from a CSV against spectra in `spectra_path`."""
    sdf, ids = load_spectra_with_ids(spectra_path)
    df = sdf.to_pandas()

    candidates_df = pd.read_csv(candidates_path)
    if not {"spectrum_id", "peptide"}.issubset(candidates_df.columns):
        raise ValueError("candidates CSV must contain at least 'spectrum_id' and 'peptide' columns")
    cand_by_spec: dict[str, list[str]] = (
        candidates_df.groupby("spectrum_id")["peptide"].apply(list).to_dict()
    )

    out_rows: list[dict] = []
    for row_idx, sid in enumerate(ids):
        cands = cand_by_spec.get(sid)
        if not cands:
            continue
        row = df.iloc[row_idx].to_dict()
        processed = processor.process_row(row)  # spectrum: (P, 2)
        spec_tensor = processed["spectra"].unsqueeze(0)
        precursor_mz = float(row["precursor_mz"])
        precursor_charge = float(row["precursor_charge"])
        precursor_mass = (precursor_mz - 1.00727647) * precursor_charge
        precursors = torch.tensor([[precursor_mass, precursor_charge, precursor_mz]], dtype=torch.float32)

        scores = casanovo_db_score(
            model,
            candidates=cands,
            spectra=spec_tensor,
            precursors=precursors,
            reduction=reduction,
        )
        for pep, sc in zip(cands, scores.tolist(), strict=True):
            out_rows.append({"spectrum_id": sid, "peptide": pep, "score": sc})

    return pd.DataFrame(out_rows)


if SPECTRA_PATH and CANDIDATES_PATH:
    rescored = rescore_from_files(SPECTRA_PATH, CANDIDATES_PATH, reduction="mean")
    rescored = rescored.sort_values(["spectrum_id", "score"], ascending=[True, False])
    rescored.to_csv(OUT_PATH, index=False)
    print(f"wrote {len(rescored)} (spectrum, candidate) scores -> {OUT_PATH}")
    rescored.head()
else:
    print("Set SPECTRA_PATH and CANDIDATES_PATH above, then re-run this cell.")

## 5. Pipeline: candidates from a FASTA database

If you don't already have a candidate list, you can generate one by digesting a FASTA file. Below is a minimal trypsin digest with proline suppression (`[KR] not followed by P`), matching the cleavage rule used in the Casanovo-DB benchmark. Then we filter candidates per spectrum by precursor-mass tolerance (ppm) before scoring.

In [ ]:
import re

AA_MONO = {
    "G": 57.02146, "A": 71.03711, "S": 87.03203, "P": 97.05276, "V": 99.06841,
    "T": 101.04768, "C": 103.00919, "L": 113.08406, "I": 113.08406, "N": 114.04293,
    "D": 115.02694, "Q": 128.05858, "K": 128.09496, "E": 129.04259, "M": 131.04049,
    "H": 137.05891, "F": 147.06841, "R": 156.10111, "Y": 163.06333, "W": 186.07931,
}
WATER = 18.01056
PROTON = 1.00727647


def trypsin_digest(
    sequence: str,
    *,
    missed_cleavages: int = 1,
    min_len: int = 6,
    max_len: int = 50,
) -> list[str]:
    """Trypsin with proline suppression. Returns peptides incl. up to `missed_cleavages`."""
    sites = [m.end() for m in re.finditer(r"[KR](?!P)", sequence)]
    sites = [0, *sites, len(sequence)]
    peptides: set[str] = set()
    for i in range(len(sites) - 1):
        for j in range(i + 1, min(i + 2 + missed_cleavages, len(sites))):
            pep = sequence[sites[i] : sites[j]]
            if min_len <= len(pep) <= max_len and set(pep).issubset(AA_MONO):
                peptides.add(pep)
    return sorted(peptides)


def peptide_mass(peptide: str) -> float:
    return WATER + sum(AA_MONO[a] for a in peptide)


def parse_fasta(path: str) -> dict[str, str]:
    seqs: dict[str, str] = {}
    name, buf = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(buf)
                name = line[1:].split()[0]
                buf = []
            elif line:
                buf.append(line)
        if name is not None:
            seqs[name] = "".join(buf)
    return seqs


def build_candidate_db(fasta_path: str, **digest_kwargs) -> list[tuple[str, float]]:
    """Return a sorted list of (peptide, monoisotopic_mass) tuples."""
    seqs = parse_fasta(fasta_path)
    peptides: set[str] = set()
    for s in seqs.values():
        peptides.update(trypsin_digest(s, **digest_kwargs))
    return sorted(((p, peptide_mass(p)) for p in peptides), key=lambda x: x[1])


def candidates_within_ppm(
    db: list[tuple[str, float]], target_mass: float, ppm: float
) -> list[str]:
    """Linear scan; for big DBs you'll want a sorted-mass binary search instead."""
    tol = target_mass * ppm * 1e-6
    return [p for p, m in db if abs(m - target_mass) <= tol]

In [ ]:
# ----- USER INPUTS -----
FASTA_PATH: str | None = None         # e.g. "data/uniprot_human.fasta"
PRECURSOR_PPM: float = 20.0
# -----------------------

if FASTA_PATH and SPECTRA_PATH:
    db = build_candidate_db(FASTA_PATH, missed_cleavages=1, min_len=6, max_len=50)
    print(f"digested DB: {len(db)} peptides")

    sdf, ids = load_spectra_with_ids(SPECTRA_PATH)
    df = sdf.to_pandas()

    out_rows: list[dict] = []
    for row_idx, sid in enumerate(ids):
        row = df.iloc[row_idx].to_dict()
        precursor_mz = float(row["precursor_mz"])
        precursor_charge = float(row["precursor_charge"])
        precursor_mass = (precursor_mz - PROTON) * precursor_charge
        cands = candidates_within_ppm(db, precursor_mass, PRECURSOR_PPM)
        if not cands:
            continue

        spec_tensor = processor.process_row(row)["spectra"].unsqueeze(0)
        precursors = torch.tensor([[precursor_mass, precursor_charge, precursor_mz]], dtype=torch.float32)
        scores = casanovo_db_score(model, cands, spectra=spec_tensor, precursors=precursors)
        # take the top-1 PSM per spectrum (database-search style)
        best = int(torch.argmax(scores).item())
        out_rows.append({
            "spectrum_id": sid,
            "peptide": cands[best],
            "score": float(scores[best]),
            "n_candidates": len(cands),
        })

    fasta_results = pd.DataFrame(out_rows).sort_values("score", ascending=False)
    fasta_results.to_csv("fasta_search_top1.csv", index=False)
    print(f"top-1 PSMs per spectrum -> fasta_search_top1.csv ({len(fasta_results)} rows)")
    fasta_results.head()
else:
    print("Set FASTA_PATH (and SPECTRA_PATH from §4) above, then re-run this cell.")

## 6. Pipeline: score against pre-computed encoder embeddings (top-k retrieval)

If you've already retrieved the top-k spectrum embeddings for each query (from a FAISS / nearest-neighbor index over `model._encoder` outputs), pass them directly here — the scorer will skip the encoder.

**Shape contract:** `spectra_embedding` is whatever `model._encoder(...)` returns: shape `(1, T, d_model)`, where `T = 1 + n_peaks` (latent + peaks) for non-flash, or `T = 2 + n_peaks` (precursor + latent + peaks) after the precursor has been concatenated. If you cached the *post-encoder, post-precursor-concat* output (i.e. the full embedding the decoder normally consumes), pass `spectra_mask=None` for flash and the corresponding mask otherwise.

In [ ]:
# ----- USER INPUTS -----
EMBEDDINGS_PATH: str | None = None    # e.g. "data/topk_embeddings.pt" (torch.save'd)
# -----------------------
#
# Expected format (flexible — adapt to your retrieval output):
#   {
#     "<spectrum_id>": {
#       "embedding": FloatTensor (T, d_model)   # one of the top-k retrieved spectrum embeddings
#       "embedding_mask": BoolTensor (T,) | None
#       "candidates": list[str]
#     }, ...
#   }

def rescore_from_embeddings(payload: dict, *, reduction: str = "mean") -> pd.DataFrame:
    rows: list[dict] = []
    for sid, item in payload.items():
        emb = item["embedding"].to(DEVICE).unsqueeze(0)  # (1, T, d)
        msk = item.get("embedding_mask")
        if msk is not None:
            msk = msk.to(DEVICE).unsqueeze(0)
        cands = item["candidates"]
        if not cands:
            continue
        scores = casanovo_db_score(
            model,
            candidates=cands,
            spectra_embedding=emb,
            spectra_mask=msk,
            reduction=reduction,
        )
        for pep, sc in zip(cands, scores.tolist(), strict=True):
            rows.append({"spectrum_id": sid, "peptide": pep, "score": sc})
    return pd.DataFrame(rows).sort_values(["spectrum_id", "score"], ascending=[True, False])


if EMBEDDINGS_PATH:
    payload = torch.load(EMBEDDINGS_PATH, map_location="cpu", weights_only=False)
    emb_results = rescore_from_embeddings(payload, reduction="mean")
    emb_results.to_csv("rescored_from_embeddings.csv", index=False)
    print(f"wrote {len(emb_results)} rows -> rescored_from_embeddings.csv")
    emb_results.head()
else:
    print("Set EMBEDDINGS_PATH above, then re-run this cell.")

## 7. Where to go next

- **FDR control via target-decoy competition (TDC).** Scores from this notebook plug straight into the standard TDC pipeline (`crema`, manual decoy generation, etc.) for peptide-level FDR estimation.
- **Percolator post-processing.** As shown in the Casanovo-DB paper, Percolator re-scoring noticeably improves calibration, especially for low-m/z precursors. Feed the score column above plus precursor m/z, charge, and (optional) peptide-length features as Percolator input.
- **Sum vs mean reduction.** Default here is the Casanovo-DB-style `mean` (geometric mean of probs). Switch to `sum` if you want the joint sequence log-probability instead — useful for direct comparison with InstaNovo's de novo confidence.
- **Batching multiple spectra.** The current driver loops one spectrum at a time and batches across candidates within a spectrum. For throughput on large datasets, batch across spectra at the encoder step and replicate per-candidate at the decoder step.